# Monte Carlo DCF in Python

## Same model and probability assumptions as the Excel classroom demo

This notebook reproduces the fictional five-year FCFF DCF from `Monte_Carlo_DCF_One_Class_Demo_EN.xlsx`.

**Learning objective:** understand that Excel and Python use the same valuation logic, while Python generates and values many simulations directly without a Data Table or dummy cell.

> **For educational purposes only. Not investment advice.**

In [ ]:
# Cell 1 — Import the minimal libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.options.display.float_format = "{:,.4f}".format
print("Libraries imported successfully.")

## 1. Assumptions

The values below exactly match the fictional Excel demo. Percentages are stored as decimals: for example, `0.05` means 5%.

In [ ]:
# Cell 2 — Base assumptions and distribution parameters
revenue_0 = 100.0

growth_base = 0.05
growth_mean = 0.05
growth_stdev = 0.02
growth_min = -0.02
growth_max = 0.12

margin_base = 0.18
margin_mean = 0.18
margin_stdev = 0.02
margin_min = 0.12
margin_max = 0.24

tax_rate = 0.25
da_pct = 0.03
capex_pct = 0.04
dnwc_pct = 0.01

wacc_base = 0.09
wacc_mean = 0.09
wacc_stdev = 0.01
wacc_min = 0.065
wacc_max = 0.12

terminal_growth_base = 0.025
terminal_growth_min = 0.01
terminal_growth_max = 0.035

assumptions = pd.DataFrame({
    "Input": [
        "Current Revenue", "Revenue Growth", "EBIT Margin", "Tax Rate",
        "D&A / Revenue", "Capex / Revenue", "Change in NWC / Revenue",
        "WACC", "Terminal Growth"
    ],
    "Base": [
        revenue_0, growth_base, margin_base, tax_rate, da_pct,
        capex_pct, dnwc_pct, wacc_base, terminal_growth_base
    ],
    "Monte Carlo treatment": [
        "Fixed", "Capped normal: mean 5%, stdev 2%, bounds -2% to 12%",
        "Capped normal: mean 18%, stdev 2%, bounds 12% to 24%", "Fixed",
        "Fixed", "Fixed", "Fixed",
        "Capped normal: mean 9%, stdev 1%, bounds 6.5% to 12%",
        "Uniform: 1% to 3.5%"
    ]
})
assumptions

## 2. Deterministic five-year FCFF DCF

The function follows the same sequence as Excel:

**Revenue → EBIT → NOPAT → FCFF → present value of FCFF → terminal value → enterprise value**

The function returns both Enterprise Value and a five-year calculation table so every step remains visible.

In [ ]:
# Cell 3 — Transparent deterministic DCF function
def calculate_ev(
    revenue_0, growth, ebit_margin, tax_rate,
    da_pct, capex_pct, dnwc_pct, wacc, terminal_growth
):
    if wacc <= terminal_growth:
        raise ValueError("WACC must be greater than terminal growth.")

    years = np.arange(1, 6)
    revenue = revenue_0 * (1 + growth) ** years
    ebit = revenue * ebit_margin
    nopat = ebit * (1 - tax_rate)
    fcff = nopat + revenue * da_pct - revenue * capex_pct - revenue * dnwc_pct
    pv_fcff = fcff / (1 + wacc) ** years

    terminal_value = fcff[-1] * (1 + terminal_growth) / (wacc - terminal_growth)
    pv_terminal_value = terminal_value / (1 + wacc) ** 5
    enterprise_value = pv_fcff.sum() + pv_terminal_value

    detail = pd.DataFrame({
        "Year": years,
        "Revenue": revenue,
        "EBIT": ebit,
        "NOPAT": nopat,
        "FCFF": fcff,
        "PV of FCFF": pv_fcff
    })
    return enterprise_value, detail, terminal_value, pv_terminal_value

print("DCF function created.")

In [ ]:
# Cell 4 — Validate the base case against Excel
base_ev, base_detail, base_tv, base_pv_tv = calculate_ev(
    revenue_0=revenue_0,
    growth=growth_base,
    ebit_margin=margin_base,
    tax_rate=tax_rate,
    da_pct=da_pct,
    capex_pct=capex_pct,
    dnwc_pct=dnwc_pct,
    wacc=wacc_base,
    terminal_growth=terminal_growth_base
)

assert np.isclose(base_ev, 201.896871, atol=0.01)
print(f"Base Enterprise Value: {base_ev:,.2f} million")
print(f"Present Value of Terminal Value: {base_pv_tv:,.2f} million")
base_detail.round(2)

> **Validation:** the deterministic Enterprise Value is approximately **201.90 million**, matching the Excel demo. This confirms that the two tools use the same DCF logic before randomness is introduced.

## 3. Random-number setup

We start with **1,000 simulations** for a direct comparison with Excel. The seed `42` makes this classroom result reproducible when the notebook is restarted and run from the top.

In [ ]:
# Cell 5 — Simulation count and reproducible random generator
n_simulations = 1_000
seed = 42
rng = np.random.default_rng(seed)

print(f"Simulations: {n_simulations:,}")
print(f"Random seed: {seed}")

## 4. Generate the uncertain inputs

`rng.normal(...)` generates normal draws. `np.clip(...)` then applies the same capping rule as Excel:

```text
MAX(lower bound, MIN(upper bound, normal draw))
```

This is a **capped normal draw**, not a triangular distribution and not a formally truncated normal distribution. Terminal Growth uses a uniform distribution because this simplified demo specifies only a lower and upper bound.

In [ ]:
# Cell 6 — Generate the same distributions used in Excel
growth = np.clip(
    rng.normal(growth_mean, growth_stdev, n_simulations),
    growth_min, growth_max
)

ebit_margin = np.clip(
    rng.normal(margin_mean, margin_stdev, n_simulations),
    margin_min, margin_max
)

wacc = np.clip(
    rng.normal(wacc_mean, wacc_stdev, n_simulations),
    wacc_min, wacc_max
)

terminal_growth = rng.uniform(
    terminal_growth_min, terminal_growth_max, n_simulations
)

random_draws = pd.DataFrame({
    "Revenue Growth": growth,
    "EBIT Margin": ebit_margin,
    "WACC": wacc,
    "Terminal Growth": terminal_growth
})

(random_draws.head()
 .style.format({column: "{:.2%}" for column in random_draws.columns}))

In [ ]:
# Cell 7 — Validate the terminal-value safety condition
assert np.all(wacc > terminal_growth), "At least one simulation has WACC <= terminal growth."

print("Validation passed: WACC is greater than Terminal Growth in every simulation.")
print(f"Minimum WACC: {wacc.min():.2%}")
print(f"Maximum Terminal Growth: {terminal_growth.max():.2%}")

## 5. Vectorized Monte Carlo DCF

Each row below represents one simulated valuation, while the five columns represent forecast Years 1 to 5. Python evaluates all simulations directly in arrays. No iteration-number column, Data Table, or dummy input cell is required.

In [ ]:
# Cell 8 — Calculate all simulated Enterprise Values
# Shape: (number of simulations, 5 forecast years)
years = np.arange(1, 6)
revenue_mc = revenue_0 * (1 + growth[:, None]) ** years[None, :]
ebit_mc = revenue_mc * ebit_margin[:, None]
nopat_mc = ebit_mc * (1 - tax_rate)
fcff_mc = (
    nopat_mc
    + revenue_mc * da_pct
    - revenue_mc * capex_pct
    - revenue_mc * dnwc_pct
)

pv_fcff_mc = fcff_mc / (1 + wacc[:, None]) ** years[None, :]
terminal_value_mc = (
    fcff_mc[:, -1] * (1 + terminal_growth)
    / (wacc - terminal_growth)
)

enterprise_value = (
    pv_fcff_mc.sum(axis=1)
    + terminal_value_mc / (1 + wacc) ** 5
)

print(f"Calculated {enterprise_value.size:,} Enterprise Values.")
print("First five values:", np.round(enterprise_value[:5], 2))

## 6. Summary statistics

The percentiles describe the simulated valuation distribution. For example, P5 is the value at or below which approximately 5% of simulated Enterprise Values fall.

In [ ]:
# Cell 9 — Same summary outputs as the Excel Results sheet
summary = pd.DataFrame({
    "Statistic": ["Mean", "Median / P50", "P5", "P25", "P75", "P95"],
    "Enterprise Value (million)": [
        np.mean(enterprise_value),
        np.median(enterprise_value),
        np.percentile(enterprise_value, 5),
        np.percentile(enterprise_value, 25),
        np.percentile(enterprise_value, 75),
        np.percentile(enterprise_value, 95)
    ]
})
summary.style.format({"Enterprise Value (million)": "{:,.2f}"})

In [ ]:
# Cell 10 — Plot the Enterprise Value distribution
p5, p50, p95 = np.percentile(enterprise_value, [5, 50, 95])

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.hist(enterprise_value, bins=35, color="#4472C4", edgecolor="white", alpha=0.9)
ax.axvline(p5, color="#C00000", linestyle="--", linewidth=2, label=f"P5: {p5:,.1f}")
ax.axvline(p50, color="#ED7D31", linestyle="--", linewidth=2, label=f"Median: {p50:,.1f}")
ax.axvline(p95, color="#70AD47", linestyle="--", linewidth=2, label=f"P95: {p95:,.1f}")
ax.set_title("Monte Carlo DCF Enterprise Value Distribution — 1,000 Simulations")
ax.set_xlabel("Enterprise Value (million)")
ax.set_ylabel("Number of Simulations")
ax.legend()
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

## Excel versus Python

| Element | Excel demo | Python notebook |
|---|---|---|
| Valuation logic | Five-year FCFF DCF | Same five-year FCFF DCF |
| Distributions | Capped normal and uniform | Same capped normal and uniform |
| Repetition | Data Table and dummy input cell | Direct array generation and calculation |
| Reproducibility | RAND changes when Excel recalculates | Fixed seed can reproduce the same sequence |
| Output | Percentiles and histogram | Same percentiles and histogram |

**Python is not more correct.** It expresses the same simulation more directly and scales the number of runs with a parameter change.

## 7. Optional scalability demonstration: 10,000 simulations

For the live comparison, change the following parameter in **Cell 5**:

```python
n_simulations = 10_000
```

Then choose **Restart Kernel and Run All**. This preserves the same seed and reruns every dependent cell in the correct order.

More simulations generally make sampling statistics more stable, but they **do not repair weak valuation assumptions or a weak DCF model**.

## Key takeaways

1. Excel and Python use the same DCF formulas and probability assumptions.
2. Python replaces the Data Table/dummy-cell repetition with direct array calculations.
3. A fixed random seed makes the classroom result reproducible.
4. Monte Carlo produces a distribution, not a uniquely correct valuation.
5. More simulations reduce sampling noise; they do not improve poor assumptions.

> **For educational purposes only. Not investment advice.**